In [1]:
# For the LLM, we recommend OpenAI with gpt-5.4-mini, but you can use any model and provider you like - 
# just adapt the client and the usage fields accordingly.

# Preparation
# First, we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure 
# everyone works with the exact same data.

# We will use gitsource for that:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

# GithubRepositoryDataReader downloads the entire repository and goes over all the files in it. 
# Because we specify allowed_extensions={"md"}, it only checks the markdown files.


In [2]:
# We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level README. 
# The lesson pages all live under a module's lessons/ folder, so filtering on /lessons/ keeps just those.

# Each file has a parse() method that returns a dictionary with its filename and content:

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
# Q1. How many lesson pages
# How many lesson pages are in the dataset?

# 24
# 72 == 72
# 240
# 720

len(documents) # 72

72

In [7]:
documents[0]["filename"] # how to extract filename

'01-agentic-rag/lessons/01-intro.md'

In [10]:
# Q2. Indexing and searching
# Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

# << How does the agentic loop keep calling the model until it stops? >>

# What's the filename of the first result?

# 01-agentic-rag/lessons/03-rag.md
# 01-agentic-rag/lessons/14-agentic-loop.md == '01-agentic-rag/lessons/14-agentic-loop.md'
# 04-evaluation/lessons/13-llm-as-judge.md
# 06-best-practices/lessons/02-hybrid-search.md

from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)
index

In [13]:
# Now search for our question from above:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    num_results=5
)

search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [ ]:
# Using RAGBase class
# Load the data and build the search index:

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

# documents = load_faq_data() # - we have done it above - 72 docs we have in total
index = build_index(documents)

# Create the assistant:

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

# Let's ask our question:

assistant.rag("How do I run Ollama locally?")